In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import timedelta
import warnings

warnings.filterwarnings('ignore')

import platform
system = platform.system()
if system == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif system == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)

# 경로 설정
DATA_PATH = Path('data')  # CSV 파일이 있는 위치
PROCESSED_PATH = Path('data/processed')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("환경 설정 완료")
print(f"데이터 경로: {DATA_PATH.absolute()}")
print(f"처리 데이터 저장 경로: {PROCESSED_PATH.absolute()}")

In [ ]:
df_patients = pd.read_csv(DATA_PATH / 'patients.csv')
print(f"✓ Patients: {len(df_patients):,}명")
df_admissions = pd.read_csv(DATA_PATH / 'admissions.csv')

# 날짜 컬럼 파싱
date_cols = ['admittime', 'dischtime', 'deathtime', 'edregtime', 'edouttime']
for col in date_cols:
    if col in df_admissions.columns:
        df_admissions[col] = pd.to_datetime(df_admissions[col])
print(f"✓ Admissions: {len(df_admissions):,}건")

df_diagnoses_icd = pd.read_csv(DATA_PATH / 'diagnoses_icd.csv')
print(f"✓ Diagnoses: {len(df_diagnoses_icd):,}건")
df_d_icd_diagnoses = pd.read_csv(DATA_PATH / 'd_icd_diagnoses.csv')
print(f"✓ ICD Dictionary: {len(df_d_icd_diagnoses):,}개 코드")
df_icustays = pd.read_csv(DATA_PATH / 'icustays.csv')
date_cols_icu = ['intime', 'outtime']
for col in date_cols_icu:
    if col in df_icustays.columns:
        df_icustays[col] = pd.to_datetime(df_icustays[col])
print(f"✓ ICU Stays: {len(df_icustays):,}건")
df_transfers = pd.read_csv(DATA_PATH / 'transfers.csv')
date_cols_trans = ['intime', 'outtime']
for col in date_cols_trans:
    if col in df_transfers.columns:
        df_transfers[col] = pd.to_datetime(df_transfers[col])
print(f"✓ Transfers: {len(df_transfers):,}건")

print("\n✓ 모든 데이터 로드 완료")

**심부전 입원 추출**

In [ ]:
print("심부전 ICD 코드 정의")
print("=" * 50)

# ICD-10: I50.x
hf_icd10 = df_d_icd_diagnoses[
    (df_d_icd_diagnoses['icd_code'].str.startswith('I50', na=False)) &
    (df_d_icd_diagnoses['icd_version'] == 10)
]['icd_code'].tolist()

# ICD-9: 428.x
hf_icd9 = df_d_icd_diagnoses[
    (df_d_icd_diagnoses['icd_code'].str.startswith('428', na=False)) &
    (df_d_icd_diagnoses['icd_version'] == 9)
]['icd_code'].tolist()

# 전체 심부전 코드
hf_codes = hf_icd10 + hf_icd9

print(f"ICD-10 심부전 코드: {len(hf_icd10)}개")
print(f"ICD-9 심부전 코드: {len(hf_icd9)}개")
print(f"총 심부전 코드: {len(hf_codes)}개")


**심부전 진단이 있는 입원 추출**

In [ ]:
print("\n" + "=" * 50)
print("심부전 진단 입원 추출")
print("=" * 50)

# 심부전 진단이 있는 hadm_id 찾기
hf_hadm_ids = df_diagnoses_icd[
    df_diagnoses_icd['icd_code'].isin(hf_codes)
]['hadm_id'].unique()

print(f"\n심부전 진단이 있는 입원: {len(hf_hadm_ids):,}건")

# admissions 테이블과 병합
df_hf_admissions = df_admissions[
    df_admissions['hadm_id'].isin(hf_hadm_ids)
].copy()

print(f"Admissions 테이블에서 매칭: {len(df_hf_admissions):,}건")
print(f"고유 환자 수: {df_hf_admissions['subject_id'].nunique():,}명")

# 샘플 확인
print("\n샘플 데이터:")
display(df_hf_admissions[['subject_id', 'hadm_id', 'admittime', 'dischtime', 
                           'admission_type', 'hospital_expire_flag']].head())


**Inclusion Criteria 적용**

In [ ]:
print("\n" + "=" * 50)
print("INCLUSION CRITERIA 적용")
print("=" * 50)

# 초기 카운트
n_initial = len(df_hf_admissions)
print(f"\n시작: {n_initial:,}건")

# 제외 건수 추적용 딕셔너리
exclusion_counts = {
    '시작': n_initial
}

# 1. 성인 환자 (≥18세)
print("\n[1] 성인 환자 필터링 (나이 ≥ 18세)")
print("-" * 80)

# 환자 정보 병합
df_hf_admissions = df_hf_admissions.merge(
    df_patients[['subject_id', 'anchor_age', 'anchor_year', 'gender']],
    on='subject_id',
    how='left'
)

# 입원 시점 나이 계산
df_hf_admissions['admit_year'] = df_hf_admissions['admittime'].dt.year
df_hf_admissions['age_at_admission'] = (
    df_hf_admissions['anchor_age'] + 
    (df_hf_admissions['admit_year'] - df_hf_admissions['anchor_year'])
)

# 18세 미만 제외
n_before = len(df_hf_admissions)
df_hf_admissions = df_hf_admissions[df_hf_admissions['age_at_admission'] >= 18]
n_after = len(df_hf_admissions)
n_excluded = n_before - n_after

print(f"18세 미만 제외: {n_excluded:,}건")
print(f"남은 입원: {n_after:,}건")
exclusion_counts['18세 미만 제외'] = n_after

# 2. 생존 퇴원
print("\n[2] 생존 퇴원 필터링")
print("-" * 80)

n_before = len(df_hf_admissions)
df_hf_admissions = df_hf_admissions[df_hf_admissions['hospital_expire_flag'] == 0]
n_after = len(df_hf_admissions)
n_excluded = n_before - n_after

print(f"원내 사망 제외: {n_excluded:,}건")
print(f"남은 입원: {n_after:,}건")
exclusion_counts['원내 사망 제외'] = n_after

# 3. 완전한 입원 기록
print("\n[3] 완전한 입원 기록 필터링")
print("-" * 80)

n_before = len(df_hf_admissions)

# 필수 컬럼 결측치 제거
required_cols = ['subject_id', 'hadm_id', 'admittime', 'dischtime']
df_hf_admissions = df_hf_admissions.dropna(subset=required_cols)

# 논리적 일관성 확인
df_hf_admissions = df_hf_admissions[
    df_hf_admissions['dischtime'] > df_hf_admissions['admittime']
]

# 입원 기간 계산
df_hf_admissions['los_days'] = (
    df_hf_admissions['dischtime'] - df_hf_admissions['admittime']
).dt.total_seconds() / (24 * 3600)

# 입원 기간 이상치 제거 (0일 초과, 365일 미만)
df_hf_admissions = df_hf_admissions[
    (df_hf_admissions['los_days'] > 0) & 
    (df_hf_admissions['los_days'] < 365)
]

n_after = len(df_hf_admissions)
n_excluded = n_before - n_after

print(f"불완전/비논리적 기록 제외: {n_excluded:,}건")
print(f"남은 입원: {n_after:,}건")
exclusion_counts['불완전 기록 제외'] = n_after

# 중간 결과
print("\n" + "=" * 80)
print("Inclusion Criteria 적용 후:")
print("=" * 80)
print(f"총 입원: {len(df_hf_admissions):,}건")
print(f"고유 환자: {df_hf_admissions['subject_id'].nunique():,}명")


**EXCLUSION CRITERIA 적용**

In [ ]:
print("\n" + "=" * 50)
print("EXCLUSION CRITERIA 적용")
print("=" * 50)

print("\n[1] 퇴원 후 30일 이내 사망 제외")
print("-" * 80)

# 사망일 정보 병합
df_hf_admissions = df_hf_admissions.merge(
    df_patients[['subject_id', 'dod']],
    on='subject_id',
    how='left',
    suffixes=('', '_patient')
)

# dod가 있는 경우만 파싱
if 'dod' in df_hf_admissions.columns:
    df_hf_admissions['dod'] = pd.to_datetime(df_hf_admissions['dod'])

n_before = len(df_hf_admissions)

# 퇴원 후 30일 이내 사망 확인
df_hf_admissions['days_to_death'] = (
    df_hf_admissions['dod'] - df_hf_admissions['dischtime']
).dt.total_seconds() / (24 * 3600)

# 30일 이내 사망 제외
df_hf_admissions = df_hf_admissions[
    (df_hf_admissions['days_to_death'].isna()) |  # 사망 정보 없음
    (df_hf_admissions['days_to_death'] > 30)      # 30일 후 사망
]

n_after = len(df_hf_admissions)
n_excluded = n_before - n_after

print(f"퇴원 후 30일 이내 사망 제외: {n_excluded:,}건")
print(f"남은 입원: {n_after:,}건")
exclusion_counts['30일 내 사망 제외'] = n_after


In [ ]:
print("\n[2] 타 병원 전원 제외")
print("-" * 80)

n_before = len(df_hf_admissions)

# 타 병원 전원 패턴
transfer_patterns = [
    'ANOTHER HOSPITAL',
    'OTHER FACILITY',
    'ACUTE HOSPITAL',
    'HOSPITAL'
]

# discharge_location 확인
if 'discharge_location' in df_hf_admissions.columns:
    # 타 병원 전원 제외
    mask_transfer = df_hf_admissions['discharge_location'].str.contains(
        '|'.join(transfer_patterns), 
        case=False, 
        na=False
    )
    
    df_hf_admissions = df_hf_admissions[~mask_transfer]
    
    n_after = len(df_hf_admissions)
    n_excluded = n_before - n_after
    
    print(f"타 병원 전원 제외: {n_excluded:,}건")
    print(f"남은 입원: {n_after:,}건")
    exclusion_counts['타 병원 전원 제외'] = n_after
else:
    print("discharge_location 컬럼 없음 - 스킵")
    exclusion_counts['타 병원 전원 제외'] = n_before

In [ ]:
print("\n[4] 산과 관련 진단 제외")
print("-" * 80)

n_before = len(df_hf_admissions)

# 산과 ICD 코드 패턴
obstetric_icd10_pattern = r'^O\d'  # O로 시작
obstetric_icd9_pattern = r'^6[3-7]\d'  # 630-679

# 산과 코드 찾기
obstetric_codes = df_d_icd_diagnoses[
    (df_d_icd_diagnoses['icd_code'].str.match(obstetric_icd10_pattern, na=False)) |
    (df_d_icd_diagnoses['icd_code'].str.match(obstetric_icd9_pattern, na=False))
]['icd_code'].tolist()

# 산과 진단이 있는 입원 찾기
obstetric_hadm_ids = df_diagnoses_icd[
    df_diagnoses_icd['icd_code'].isin(obstetric_codes)
]['hadm_id'].unique()

# 제외
df_hf_admissions = df_hf_admissions[
    ~df_hf_admissions['hadm_id'].isin(obstetric_hadm_ids)
]

n_after = len(df_hf_admissions)
n_excluded = n_before - n_after

print(f"산과 관련 진단 제외: {n_excluded:,}건")
print(f"남은 입원: {n_after:,}건")
exclusion_counts['산과 진단 제외'] = n_after

# 중간 결과
print("\n" + "=" * 80)
print("Exclusion Criteria 적용 후:")
print("=" * 80)
print(f"총 입원: {len(df_hf_admissions):,}건")
print(f"고유 환자: {df_hf_admissions['subject_id'].nunique():,}명")

**재입원 데이터 라벨링**

index admission에 대해 30일 이내 재입원 여부 라벨링. 이는 머신러닝 모델의 타겟 변수가 된다.

In [ ]:
# 환자별로 입원 시간순 정렬
df_hf_admissions = df_hf_admissions.sort_values(['subject_id', 'admittime'])

# 환자별로 다음 입원 시간 계산
df_hf_admissions['next_admittime'] = df_hf_admissions.groupby('subject_id')['admittime'].shift(-1)

# 재입원까지의 일수 계산
df_hf_admissions['days_to_readmission'] = (
    df_hf_admissions['next_admittime'] - df_hf_admissions['dischtime']
).dt.total_seconds() / (24 * 3600)

# 30일 이내 재입원 = 1, 아니면 = 0
df_hf_admissions['readmission_30d'] = (
    (df_hf_admissions['days_to_readmission'].notna()) &  # 다음 입원이 있고
    (df_hf_admissions['days_to_readmission'] <= 30) &    # 30일 이내이고
    (df_hf_admissions['days_to_readmission'] > 0)        # 양수(동일날 재입원 제외)
).astype(int)

In [ ]:
# 1. 재입원율 계산
readmission_rate = df_hf_admissions['readmission_30d'].mean() * 100
print(f"30일 재입원율: {readmission_rate:.2f}%")

# 2. 재입원까지의 일수 분포
df_readmitted = df_hf_admissions[df_hf_admissions['readmission_30d'] == 1]
print(f"\n재입원 통계:")
print(df_readmitted['days_to_readmission'].describe())

# 3. 시각화
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 4))
plt.hist(df_readmitted['days_to_readmission'], bins=30, edgecolor='black')
plt.xlabel('재입원까지의 일수')
plt.ylabel('빈도')
plt.title('30일 이내 재입원 시점 분포')
plt.show()

In [ ]:
# 최소 재입원 간격 확인
very_short = df_hf_admissions[
    (df_hf_admissions['readmission_30d'] == 1) & 
    (df_hf_admissions['days_to_readmission'] < 1)
]
print(f"1일 이내 재입원: {len(very_short)}건")

In [ ]:
# 1일 이내 재입원의 경우 제외

# 현재 상태 확인
n_before = len(df_hf_admissions)
n_readmit_before = df_hf_admissions['readmission_30d'].sum()

# 1일 미만 재입원 건수 확인
very_short_readmit = (
    (df_hf_admissions['readmission_30d'] == 1) & 
    (df_hf_admissions['days_to_readmission'] < 1)
).sum()

print(f"\n1일 미만 재입원: {very_short_readmit}건")
print(f"제외 전 재입원: {n_readmit_before}건 ({n_readmit_before/n_before*100:.2f}%)")

# 1일 미만 재입원 제외
# - 재입원이 아닌 경우(readmission_30d == 0)는 유지
# - 재입원인 경우(readmission_30d == 1)는 1일 이상만 유지
df_hf_admissions = df_hf_admissions[
    (df_hf_admissions['readmission_30d'] == 0) |
    (df_hf_admissions['days_to_readmission'] >= 1)
]

# 제외 후 상태
n_after = len(df_hf_admissions)
n_readmit_after = df_hf_admissions['readmission_30d'].sum()
n_excluded = n_before - n_after

print(f"\n제외된 입원: {n_excluded}건")
print(f"남은 입원: {n_after:,}건")
print(f"제외 후 재입원: {n_readmit_after}건 ({n_readmit_after/n_after*100:.2f}%)")
print(f"고유 환자: {df_hf_admissions['subject_id'].nunique():,}명")

# 재입원 통계 업데이트
print("\n업데이트된 재입원 통계:")
df_readmitted = df_hf_admissions[df_hf_admissions['readmission_30d'] == 1]
print(df_readmitted['days_to_readmission'].describe())

# 시각화 업데이트
plt.figure(figsize=(12, 5))
plt.hist(df_readmitted['days_to_readmission'], bins=29, edgecolor='black', alpha=0.7)
plt.xlabel('재입원까지의 일수', fontsize=12)
plt.ylabel('빈도', fontsize=12)
plt.title('30일 이내 재입원 시점 분포 (1일 이상)', fontsize=14)
plt.axvline(x=7, color='red', linestyle='--', linewidth=2, label='7일 (조기 재입원)')
plt.axvline(x=df_readmitted['days_to_readmission'].median(), 
            color='green', linestyle='--', linewidth=2, 
            label=f"중앙값 ({df_readmitted['days_to_readmission'].median():.1f}일)")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PROCESSED_PATH / '30day_readmission_distribution_filtered.png', dpi=300, bbox_inches='tight')
plt.show()

# Exclusion counts 업데이트
exclusion_counts['1일 미만 재입원 제외'] = n_after

**중복 입원 처리**
1. Multiple readmissions: 한 환자가 30일 내 여러 번 재입원 → 첫 번째만 카운트
2. Index admission overlap: 재입원으로 사용된 입원은 독립적인 index admission으로 사용 불가

In [ ]:
n_before = len(df_hf_admissions)
n_readmit_before = df_hf_admissions['readmission_30d'].sum()

print(f"\n처리 전:")
print(f"총 입원: {n_before:,}건")
print(f"재입원: {n_readmit_before:,}건 ({n_readmit_before/n_before*100:.2f}%)")
print(f"고유 환자: {df_hf_admissions['subject_id'].nunique():,}명")

In [ ]:
print("재입원으로 사용된 입원 식별")

# 다음 입원의 hadm_id를 현재 행에 추가 (아직 없다면)
if 'next_hadm_id' not in df_hf_admissions.columns:
    df_hf_admissions['next_hadm_id'] = df_hf_admissions.groupby('subject_id')['hadm_id'].shift(-1)

# 재입원인 경우, next_hadm_id가 실제 재입원의 hadm_id
readmission_hadm_ids = df_hf_admissions[
    df_hf_admissions['readmission_30d'] == 1
]['next_hadm_id'].dropna().values

print(f"재입원으로 사용된 입원 수: {len(readmission_hadm_ids):,}건")

In [ ]:
print("재입원으로 사용된 입원을 index admission에서 제외")

# 재입원으로 사용된 입원만 제거
mask_readmission_as_index = df_hf_admissions['hadm_id'].isin(readmission_hadm_ids)
n_excluded_as_readmission = mask_readmission_as_index.sum()

print(f"제외될 입원: {n_excluded_as_readmission:,}건")

# 제외 (재입원 라벨은 그대로 유지!)
df_hf_admissions_final = df_hf_admissions[~mask_readmission_as_index].copy()

print(f"남은 입원: {len(df_hf_admissions_final):,}건")

In [ ]:
n_after = len(df_hf_admissions_final)
n_readmit_after = df_hf_admissions_final['readmission_30d'].sum()
n_patients_after = df_hf_admissions_final['subject_id'].nunique()

print(f"\n최종 코호트:")
print(f"총 입원: {n_after:,}건 (제외: {n_before - n_after:,}건)")
print(f"재입원: {n_readmit_after:,}건 ({n_readmit_after/n_after*100:.2f}%)")
print(f"비재입원: {n_after - n_readmit_after:,}건 ({(n_after - n_readmit_after)/n_after*100:.2f}%)")
print(f"고유 환자: {n_patients_after:,}명")

# 재입원 통계
print("\n재입원 통계:")
df_readmitted = df_hf_admissions_final[df_hf_admissions_final['readmission_30d'] == 1]
if len(df_readmitted) > 0:
    print(df_readmitted['days_to_readmission'].describe())
    
    print(f"\n재입원 시점 분석:")
    print(f"  - 7일 이내: {(df_readmitted['days_to_readmission'] <= 7).sum():,}건 "
          f"({(df_readmitted['days_to_readmission'] <= 7).sum()/len(df_readmitted)*100:.1f}%)")
    print(f"  - 14일 이내: {(df_readmitted['days_to_readmission'] <= 14).sum():,}건 "
          f"({(df_readmitted['days_to_readmission'] <= 14).sum()/len(df_readmitted)*100:.1f}%)")
else:
    print("재입원 없음")

# exclusion_counts 업데이트
exclusion_counts['중복 입원 제외'] = n_after

# 최종 데이터프레임 업데이트
df_hf_admissions = df_hf_admissions_final.copy()

**최종 코호트 정의**

In [ ]:
print("[최종 코호트 정의]")
print("-" * 50)
print(f"총 입원 건수: {len(df_hf_admissions):,}건")
print(f"재입원: {df_hf_admissions['readmission_30d'].sum():,}건 ({df_hf_admissions['readmission_30d'].mean()*100:.2f}%)")
print(f"비재입원: {(df_hf_admissions['readmission_30d']==0).sum():,}건 ({(df_hf_admissions['readmission_30d']==0).mean()*100:.2f}%)")
print(f"고유 환자: {df_hf_admissions['subject_id'].nunique():,}명")
#print(f"데이터 기간: {df_hf_admissions['admittime'].min()} ~ {df_hf_admissions['admittime'].max()}")

In [ ]:
print("\n[데이터 품질 검증]")
print("-" * 50)

print("\n1. 필수 컬럼 결측치 확인:")
required_cols = ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'readmission_30d']
missing_check = df_hf_admissions[required_cols].isnull().sum()

for col in required_cols:
    n_missing = missing_check[col]
    pct_missing = n_missing / len(df_hf_admissions) * 100
    status = "✓" if n_missing == 0 else "⚠"
    print(f"  {status} {col}: {n_missing}건 결측 ({pct_missing:.2f}%)")

# 전체 결측치 확인
total_missing = df_hf_admissions.isnull().sum().sum()
print(f"\n  총 결측치: {total_missing:,}개")

In [ ]:
print("\n2. 논리적 일관성 확인:")

# 입원/퇴원 시간 순서
invalid_dates = df_hf_admissions[
    df_hf_admissions['dischtime'] <= df_hf_admissions['admittime']
]
print(f"  ✓ 퇴원 < 입원 오류: {len(invalid_dates)}건")

# 입원 기간 범위
los_stats = df_hf_admissions['los_days'].describe()
print(f"  ✓ 입원 기간 (LOS):")
print(f"    - 평균: {los_stats['mean']:.1f}일")
print(f"    - 중앙값: {los_stats['50%']:.1f}일")
print(f"    - 범위: {los_stats['min']:.1f} ~ {los_stats['max']:.1f}일")

# 재입원 일수 범위 (재입원인 경우만)
df_readmit = df_hf_admissions[df_hf_admissions['readmission_30d'] == 1]
if len(df_readmit) > 0:
    invalid_readmit_days = df_readmit[
        (df_readmit['days_to_readmission'] < 1) | 
        (df_readmit['days_to_readmission'] > 30)
    ]
    print(f"  ✓ 재입원 일수 범위 오류 (1-30일 밖): {len(invalid_readmit_days)}건")

# 중복 hadm_id 확인
duplicate_hadm = df_hf_admissions['hadm_id'].duplicated().sum()
print(f"  ✓ 중복 hadm_id: {duplicate_hadm}건")

In [ ]:
print("\n3. 환자 및 입원 특성:")

# 환자별 입원 횟수
admissions_per_patient = df_hf_admissions.groupby('subject_id').size()
print(f"  환자당 입원 횟수:")
print(f"    - 평균: {admissions_per_patient.mean():.2f}회")
print(f"    - 중앙값: {admissions_per_patient.median():.0f}회")
print(f"    - 최대: {admissions_per_patient.max()}회")

# 입원 유형 분포 (있는 경우)
if 'admission_type' in df_hf_admissions.columns:
    print(f"\n  입원 유형 분포:")
    admission_types = df_hf_admissions['admission_type'].value_counts()
    for adm_type, count in admission_types.head(5).items():
        print(f"    - {adm_type}: {count:,}건 ({count/len(df_hf_admissions)*100:.1f}%)")

# 재입원 vs 비재입원 비교
print(f"\n  재입원 vs 비재입원 입원 기간(LOS) 비교:")
los_readmit = df_hf_admissions[df_hf_admissions['readmission_30d']==1]['los_days'].mean()
los_no_readmit = df_hf_admissions[df_hf_admissions['readmission_30d']==0]['los_days'].mean()
print(f"    - 재입원 그룹: {los_readmit:.1f}일")
print(f"    - 비재입원 그룹: {los_no_readmit:.1f}일")
print(f"    - 차이: {los_readmit - los_no_readmit:+.1f}일")

In [ ]:
print("\n[Exclusion Criteria 흐름 요약]")
print("=" * 50)

# Exclusion 단계별 요약
exclusion_summary = pd.DataFrame({
    '단계': [
        '심부전 입원 추출',
        'Inclusion: 18세 이상',
        'Inclusion: 원내 사망 제외',
        'Inclusion: 완전한 기록',
        'Exclusion: 30일 내 사망',
        'Exclusion: 타 병원 전원',
        'Exclusion: 산과 진단',
        '재입원 라벨링 (1일 미만 제외)',
        '중복 입원 제거',
        '최종 코호트'
    ],
    '입원 건수': [
        exclusion_counts['시작'],
        exclusion_counts['18세 미만 제외'],
        exclusion_counts['원내 사망 제외'],
        exclusion_counts['불완전 기록 제외'],
        exclusion_counts['30일 내 사망 제외'],
        exclusion_counts['타 병원 전원 제외'],
        exclusion_counts['산과 진단 제외'],
        exclusion_counts['1일 미만 재입원 제외'],
        exclusion_counts['중복 입원 제외'],
        len(df_hf_admissions)
    ]
})

# 제외 건수 계산
exclusion_summary['제외 건수'] = 0
for i in range(1, len(exclusion_summary)):
    excluded = exclusion_summary.loc[i-1, '입원 건수'] - exclusion_summary.loc[i, '입원 건수']
    exclusion_summary.loc[i, '제외 건수'] = int(abs(excluded))

# 포맷팅
exclusion_summary['입원 건수_display'] = exclusion_summary['입원 건수'].apply(lambda x: f"{x:,}")
exclusion_summary['제외 건수_display'] = exclusion_summary['제외 건수'].apply(
    lambda x: f"{x:,}" if x > 0 else "-"
)

# 표시용
display_df = exclusion_summary[['단계', '입원 건수_display', '제외 건수_display']].copy()
display_df.columns = ['단계', '입원 건수', '제외 건수']

print("\n")
print(display_df.to_string(index=False))

# 통계 요약
print("\n" + "=" * 80)
print("전체 제외 요약:")
total_excluded = exclusion_summary['입원 건수'].iloc[0] - exclusion_summary['입원 건수'].iloc[-1]
exclusion_rate = (total_excluded / exclusion_summary['입원 건수'].iloc[0]) * 100
print(f"  초기 입원: {exclusion_summary['입원 건수'].iloc[0]:,}건")
print(f"  최종 코호트: {exclusion_summary['입원 건수'].iloc[-1]:,}건")
print(f"  총 제외: {total_excluded:,}건 ({exclusion_rate:.1f}%)")

In [ ]:
print("\n[최종 코호트 저장]")
print("=" * 50)

# 저장할 컬럼 선택
save_columns = [
    'subject_id', 'hadm_id', 
    'admittime', 'dischtime', 
    'admission_type', 'discharge_location',
    'los_days', 
    'readmission_30d', 'days_to_readmission',
    'hospital_expire_flag'
]

# 존재하는 컬럼만 선택
available_columns = [col for col in save_columns if col in df_hf_admissions.columns]
df_to_save = df_hf_admissions[available_columns].copy()

# CSV 저장
output_file = PROCESSED_PATH / 'final_cohort.csv'
df_to_save.to_csv(output_file, index=False)

print(f"✓ 최종 코호트 저장 완료: {output_file}")
print(f"  - 파일 크기: {output_file.stat().st_size / 1024 / 1024:.2f} MB")
print(f"  - 행 수: {len(df_to_save):,}건")
print(f"  - 열 수: {len(df_to_save.columns)}개")
print(f"  - 컬럼: {', '.join(df_to_save.columns)}")

In [ ]:
# Exclusion 요약 저장
exclusion_file = PROCESSED_PATH / 'exclusion_summary.csv'
exclusion_summary.to_csv(exclusion_file, index=False)

In [ ]:
print("\n[최종 통계 시각화]")
print("=" * 50)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 그래프 1: 재입원 vs 비재입원
ax1 = axes[0, 0]
readmit_counts = df_hf_admissions['readmission_30d'].value_counts()
colors = ['#2ecc71', '#e74c3c']
labels = ['비재입원', '재입원']
ax1.pie(readmit_counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('재입원 vs 비재입원 분포', fontsize=12, fontweight='bold')

# 그래프 2: 입원 기간(LOS) 분포
ax2 = axes[0, 1]
ax2.hist(df_hf_admissions['los_days'], bins=50, color='#3498db', alpha=0.7, edgecolor='black')
ax2.axvline(df_hf_admissions['los_days'].median(), color='red', linestyle='--', linewidth=2, label=f"중앙값: {df_hf_admissions['los_days'].median():.1f}일")
ax2.set_xlabel('입원 기간 (일)', fontsize=11)
ax2.set_ylabel('빈도', fontsize=11)
ax2.set_title('입원 기간 분포', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# 그래프 3: 재입원까지의 일수 분포
ax3 = axes[1, 0]
df_readmit = df_hf_admissions[df_hf_admissions['readmission_30d'] == 1]
if len(df_readmit) > 0:
    ax3.hist(df_readmit['days_to_readmission'], bins=29, color='#e74c3c', alpha=0.7, edgecolor='black')
    ax3.axvline(7, color='orange', linestyle='--', linewidth=2, label='7일 (조기)')
    ax3.axvline(df_readmit['days_to_readmission'].median(), color='green', linestyle='--', linewidth=2, label=f"중앙값: {df_readmit['days_to_readmission'].median():.1f}일")
    ax3.set_xlabel('재입원까지의 일수', fontsize=11)
    ax3.set_ylabel('빈도', fontsize=11)
    ax3.set_title('재입원 시점 분포 (1일 이상)', fontsize=12, fontweight='bold')
    ax3.legend()
    ax3.grid(axis='y', alpha=0.3)

# 그래프 4: 환자별 입원 횟수
ax4 = axes[1, 1]
admissions_per_patient = df_hf_admissions.groupby('subject_id').size()
admission_counts = admissions_per_patient.value_counts().sort_index().head(10)
ax4.bar(admission_counts.index, admission_counts.values, color='#9b59b6', alpha=0.7, edgecolor='black')
ax4.set_xlabel('입원 횟수', fontsize=11)
ax4.set_ylabel('환자 수', fontsize=11)
ax4.set_title('환자별 입원 횟수 분포', fontsize=12, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PROCESSED_PATH / 'part7_final_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("\n[재입원 vs 비재입원 특성 비교]")
print("=" * 50)

# 두 그룹 분리
df_readmit_group = df_hf_admissions[df_hf_admissions['readmission_30d'] == 1]
df_no_readmit_group = df_hf_admissions[df_hf_admissions['readmission_30d'] == 0]

print(f"\n재입원 그룹 (n={len(df_readmit_group):,}):")
print(f"  - 평균 입원 기간: {df_readmit_group['los_days'].mean():.2f}일 (SD: {df_readmit_group['los_days'].std():.2f})")
print(f"  - 중앙값 입원 기간: {df_readmit_group['los_days'].median():.2f}일")

print(f"\n비재입원 그룹 (n={len(df_no_readmit_group):,}):")
print(f"  - 평균 입원 기간: {df_no_readmit_group['los_days'].mean():.2f}일 (SD: {df_no_readmit_group['los_days'].std():.2f})")
print(f"  - 중앙값 입원 기간: {df_no_readmit_group['los_days'].median():.2f}일")

# 통계적 차이
los_diff = df_readmit_group['los_days'].mean() - df_no_readmit_group['los_days'].mean()
print(f"\n입원 기간 차이: {los_diff:+.2f}일")

**+) CONSORT 다이어그램**

In [ ]:
# Exclusion 데이터 정리
consort_data = {
    'initial': exclusion_counts['시작'],
    'excluded_age': 0,  # 18세 미만 제외 없음
    'after_age': exclusion_counts['18세 미만 제외'],
    
    'excluded_death_hospital': exclusion_counts['18세 미만 제외'] - exclusion_counts['원내 사망 제외'],
    'after_death_hospital': exclusion_counts['원내 사망 제외'],
    
    'excluded_incomplete': exclusion_counts['원내 사망 제외'] - exclusion_counts['불완전 기록 제외'],
    'after_incomplete': exclusion_counts['불완전 기록 제외'],
    
    'excluded_death_30d': exclusion_counts['불완전 기록 제외'] - exclusion_counts['30일 내 사망 제외'],
    'after_death_30d': exclusion_counts['30일 내 사망 제외'],
    
    'excluded_transfer': exclusion_counts['30일 내 사망 제외'] - exclusion_counts['타 병원 전원 제외'],
    'after_transfer': exclusion_counts['타 병원 전원 제외'],
    
    'excluded_obstetric': exclusion_counts['타 병원 전원 제외'] - exclusion_counts['산과 진단 제외'],
    'after_obstetric': exclusion_counts['산과 진단 제외'],
    
    'excluded_1day': exclusion_counts['산과 진단 제외'] - exclusion_counts['1일 미만 재입원 제외'],
    'after_1day': exclusion_counts['1일 미만 재입원 제외'],
    
    'excluded_duplicate': exclusion_counts['1일 미만 재입원 제외'] - exclusion_counts['중복 입원 제외'],
    
    'final_admissions': len(df_hf_admissions),
    'final_patients': df_hf_admissions['subject_id'].nunique(),
    'readmissions': df_hf_admissions['readmission_30d'].sum(),
    'no_readmissions': (df_hf_admissions['readmission_30d'] == 0).sum()
}

In [ ]:
fig, ax = plt.subplots(figsize=(14, 18))
ax.axis('off')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# 박스 스타일 정의
box_style = dict(boxstyle='round,pad=0.5', facecolor='#E3F2FD', edgecolor='#1976D2', linewidth=2)
excluded_style = dict(boxstyle='round,pad=0.4', facecolor='#FFCDD2', edgecolor='#C62828', linewidth=1.5)
final_style = dict(boxstyle='round,pad=0.5', facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=3)
outcome_readmit = dict(boxstyle='round,pad=0.4', facecolor='#FFCDD2', edgecolor='#C62828', linewidth=2)
outcome_no = dict(boxstyle='round,pad=0.4', facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=2)

# 위치 설정
x_center = 0.38
x_excluded = 0.75
box_width = 0.22
excluded_width = 0.18

# Y 위치 (더 넓은 간격)
y_start = 0.96
y_step = 0.082
y_positions = [y_start - i * y_step for i in range(10)]

# 화살표 스타일
arrow_props = dict(arrowstyle='->', lw=2.5, color='#1976D2')
excluded_arrow_props = dict(arrowstyle='->', lw=2, color='#C62828', linestyle='--')

# 제목
ax.text(0.5, 0.99, 'CONSORT Flow Diagram', 
        ha='center', va='top', fontsize=16, fontweight='bold')
ax.text(0.5, 0.975, 'Heart Failure 30-Day Readmission Study', 
        ha='center', va='top', fontsize=13)

# 1. 초기 코호트
y_idx = 0
ax.text(x_center, y_positions[y_idx], 
        f'Heart Failure Hospitalizations\nIdentified from MIMIC-IV\n(n = {consort_data["initial"]:,})',
        ha='center', va='center', fontsize=11, fontweight='bold', bbox=box_style)
ax.annotate('', xy=(x_center, y_positions[y_idx+1]+0.025), xytext=(x_center, y_positions[y_idx]-0.03),
            arrowprops=arrow_props)

# 2. 원내 사망 제외
y_idx = 1
ax.text(x_center, y_positions[y_idx],
        f'After excluding in-hospital deaths\n(n = {consort_data["after_death_hospital"]:,})',
        ha='center', va='center', fontsize=10, bbox=box_style)
ax.text(x_excluded, y_positions[y_idx],
        f'Excluded:\nIn-hospital deaths\n(n = {consort_data["excluded_death_hospital"]:,})',
        ha='center', va='center', fontsize=9, bbox=excluded_style)
ax.annotate('', xy=(x_excluded-excluded_width/2-0.01, y_positions[y_idx]), 
            xytext=(x_center+box_width/2+0.01, y_positions[y_idx]),
            arrowprops=excluded_arrow_props)
ax.annotate('', xy=(x_center, y_positions[y_idx+1]+0.025), xytext=(x_center, y_positions[y_idx]-0.03),
            arrowprops=arrow_props)

# 3. 불완전한 기록 제외
y_idx = 2
ax.text(x_center, y_positions[y_idx],
        f'After excluding incomplete records\n(n = {consort_data["after_incomplete"]:,})',
        ha='center', va='center', fontsize=10, bbox=box_style)
ax.text(x_excluded, y_positions[y_idx],
        f'Excluded:\nIncomplete records\n(n = {consort_data["excluded_incomplete"]:,})',
        ha='center', va='center', fontsize=9, bbox=excluded_style)
ax.annotate('', xy=(x_excluded-excluded_width/2-0.01, y_positions[y_idx]), 
            xytext=(x_center+box_width/2+0.01, y_positions[y_idx]),
            arrowprops=excluded_arrow_props)
ax.annotate('', xy=(x_center, y_positions[y_idx+1]+0.025), xytext=(x_center, y_positions[y_idx]-0.03),
            arrowprops=arrow_props)

# 4. 30일 내 사망 제외
y_idx = 3
ax.text(x_center, y_positions[y_idx],
        f'After excluding deaths within 30 days\n(n = {consort_data["after_death_30d"]:,})',
        ha='center', va='center', fontsize=10, bbox=box_style)
ax.text(x_excluded, y_positions[y_idx],
        f'Excluded:\nDeaths ≤30 days\npost-discharge\n(n = {consort_data["excluded_death_30d"]:,})',
        ha='center', va='center', fontsize=9, bbox=excluded_style)
ax.annotate('', xy=(x_excluded-excluded_width/2-0.01, y_positions[y_idx]), 
            xytext=(x_center+box_width/2+0.01, y_positions[y_idx]),
            arrowprops=excluded_arrow_props)
ax.annotate('', xy=(x_center, y_positions[y_idx+1]+0.025), xytext=(x_center, y_positions[y_idx]-0.03),
            arrowprops=arrow_props)

# 5. 타 병원 전원 제외
y_idx = 4
ax.text(x_center, y_positions[y_idx],
        f'After excluding transfers\n(n = {consort_data["after_transfer"]:,})',
        ha='center', va='center', fontsize=10, bbox=box_style)
ax.text(x_excluded, y_positions[y_idx],
        f'Excluded:\nTransfers to other\nacute care hospitals\n(n = {consort_data["excluded_transfer"]:,})',
        ha='center', va='center', fontsize=9, bbox=excluded_style)
ax.annotate('', xy=(x_excluded-excluded_width/2-0.01, y_positions[y_idx]), 
            xytext=(x_center+box_width/2+0.01, y_positions[y_idx]),
            arrowprops=excluded_arrow_props)
ax.annotate('', xy=(x_center, y_positions[y_idx+1]+0.025), xytext=(x_center, y_positions[y_idx]-0.03),
            arrowprops=arrow_props)

# 6. 산과 진단 제외
y_idx = 5
ax.text(x_center, y_positions[y_idx],
        f'After excluding obstetric admissions\n(n = {consort_data["after_obstetric"]:,})',
        ha='center', va='center', fontsize=10, bbox=box_style)
ax.text(x_excluded, y_positions[y_idx],
        f'Excluded:\nObstetric-related\nadmissions\n(n = {consort_data["excluded_obstetric"]:,})',
        ha='center', va='center', fontsize=9, bbox=excluded_style)
ax.annotate('', xy=(x_excluded-excluded_width/2-0.01, y_positions[y_idx]), 
            xytext=(x_center+box_width/2+0.01, y_positions[y_idx]),
            arrowprops=excluded_arrow_props)
ax.annotate('', xy=(x_center, y_positions[y_idx+1]+0.025), xytext=(x_center, y_positions[y_idx]-0.03),
            arrowprops=arrow_props)

# 7. 1일 미만 재입원 제외
y_idx = 6
ax.text(x_center, y_positions[y_idx],
        f'After excluding readmissions <24 hours\n(n = {consort_data["after_1day"]:,})',
        ha='center', va='center', fontsize=10, bbox=box_style)
ax.text(x_excluded, y_positions[y_idx],
        f'Excluded:\nReadmissions\n<24 hours\n(n = {consort_data["excluded_1day"]:,})',
        ha='center', va='center', fontsize=9, bbox=excluded_style)
ax.annotate('', xy=(x_excluded-excluded_width/2-0.01, y_positions[y_idx]), 
            xytext=(x_center+box_width/2+0.01, y_positions[y_idx]),
            arrowprops=excluded_arrow_props)
ax.annotate('', xy=(x_center, y_positions[y_idx+1]+0.025), xytext=(x_center, y_positions[y_idx]-0.03),
            arrowprops=arrow_props)

# 8. 중복 입원 제외
y_idx = 7
ax.text(x_center, y_positions[y_idx],
        f'After excluding duplicate admissions\n(n = {consort_data["after_1day"]:,})',
        ha='center', va='center', fontsize=10, bbox=box_style)
ax.text(x_excluded, y_positions[y_idx],
        f'Excluded:\nAdmissions used as\nreadmission outcomes\n(n = {consort_data["excluded_duplicate"]:,})',
        ha='center', va='center', fontsize=9, bbox=excluded_style)
ax.annotate('', xy=(x_excluded-excluded_width/2-0.01, y_positions[y_idx]), 
            xytext=(x_center+box_width/2+0.01, y_positions[y_idx]),
            arrowprops=excluded_arrow_props)
ax.annotate('', xy=(x_center, y_positions[y_idx+1]+0.025), xytext=(x_center, y_positions[y_idx]-0.03),
            arrowprops=arrow_props)

# 9. 최종 코호트
y_idx = 8
ax.text(x_center, y_positions[y_idx],
        f'Final Cohort\nIndex Hospitalizations: {consort_data["final_admissions"]:,}\n' +
        f'Unique Patients: {consort_data["final_patients"]:,}',
        ha='center', va='center', fontsize=11, fontweight='bold', bbox=final_style)
ax.annotate('', xy=(x_center, y_positions[y_idx+1]+0.025), xytext=(x_center, y_positions[y_idx]-0.035),
            arrowprops=arrow_props)

# 10. 최종 결과
y_idx = 9
x_left = 0.28
x_right = 0.48

ax.text(x_left, y_positions[y_idx],
        f'30-day\nReadmission\n(n = {consort_data["readmissions"]:,})\n16.1%',
        ha='center', va='center', fontsize=10, fontweight='bold', bbox=outcome_readmit)

ax.text(x_right, y_positions[y_idx],
        f'No\nReadmission\n(n = {consort_data["no_readmissions"]:,})\n83.9%',
        ha='center', va='center', fontsize=10, fontweight='bold', bbox=outcome_no)

# 최종 결과로 가는 화살표
ax.annotate('', xy=(x_left, y_positions[y_idx]+0.025), 
            xytext=(x_center-0.05, y_positions[y_idx-1]-0.035),
            arrowprops=arrow_props)
ax.annotate('', xy=(x_right, y_positions[y_idx]+0.025), 
            xytext=(x_center+0.05, y_positions[y_idx-1]-0.035),
            arrowprops=arrow_props)

plt.tight_layout()
plt.savefig(PROCESSED_PATH / 'consort_diagram.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show() 

In [ ]:
print("\n[CONSORT 요약 통계]")
print("=" * 50)

# 요약 테이블
consort_summary = pd.DataFrame({
    'Stage': [
        'Initial Cohort',
        'Final Cohort',
        'Total Excluded',
        '',
        'Readmission',
        'No Readmission'
    ],
    'N': [
        consort_data['initial'],
        consort_data['final_admissions'],
        consort_data['initial'] - consort_data['final_admissions'],
        '',
        consort_data['readmissions'],
        consort_data['no_readmissions']
    ],
    'Percentage': [
        '100%',
        f"{consort_data['final_admissions']/consort_data['initial']*100:.1f}%",
        f"{(consort_data['initial'] - consort_data['final_admissions'])/consort_data['initial']*100:.1f}%",
        '',
        f"{consort_data['readmissions']/consort_data['final_admissions']*100:.1f}%",
        f"{consort_data['no_readmissions']/consort_data['final_admissions']*100:.1f}%"
    ]
})

print("\n")
print(consort_summary.to_string(index=False))